# 07 — Fuel and epoch metering decomposition

Canonical analysis uses the four parsed modes `neither`, `fuel-only`, `epoch-only`, and `both`. N is independent runs. Differences, ratios, bootstrap intervals, and Cliff's delta are computed against `neither`; filenames do not define effective metering.


In [ ]:
import os, re
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.canonical import metering_table
from wafer_analysis.focused import evidence_label, pending_record, percentile_rows
from wafer_analysis.paths import resolve_analysis_batch
from wafer_analysis.plots import save_figure
from wafer_analysis.tables import save_table

batch, canonical=resolve_analysis_batch('e-perf-7',os.environ.get('E_PERF_7_DIR'))
raw=pd.DataFrame() if batch is None else percentile_rows(batch)
conditions=['neither','fuel-only','epoch-only','both']
if canonical:
    records=[]
    for row in raw.to_dict('records'):
        match=re.search(r'run-(\d+)',str(row['run']))
        if match is None: raise ValueError(f"malformed canonical run name: {row['run']}")
        records.append({**row,'run_index':int(match.group(1))})
    out=metering_table(records)
else:
    rows=[]
    for condition in conditions:
        values=raw[raw.condition==condition] if not raw.empty else raw
        if values.empty: rows.append(pending_record(condition,'no passed percentile leaf','nanoseconds'))
        else: rows.append({'condition':condition,'status':'READY','N_runs':len(values),'median_p95_ns':values.p95_ns.median(),'units':'nanoseconds','estimator':'median run p95','uncertainty':'descriptive only','claim_boundary':'diagnostic ablation only','thesis_evidence':False})
    out=pd.DataFrame(rows)
print(evidence_label(int(out.get('N_runs',pd.Series(dtype=int)).sum()),'nanoseconds',canonical)); display(out)
ready=out[out.get('status',pd.Series(['READY']*len(out))).eq('READY')] if not out.empty else out
if not ready.empty:
    fig,ax=plt.subplots(); ax.bar(ready['condition'],ready['median_p95_ns']/1e3); ax.set_ylabel('Median run p95 (µs)'); ax.set_title('Metering decomposition — run-level')
    output=os.environ.get('WAFER_ANALYSIS_OUTPUT_DIR')
    if output: save_figure(fig,'e-perf-7/metering-decomposition',output); save_table(out,'e-perf-7-metering-decomposition',output)
